In [3]:
using Random
using Statistics
using Printf

# Set a random seed for reproducibility
Random.seed!(0)

# --- Data Generation Function ---
function spiral_data(samples::Int, classes::Int)
    X = zeros(Float32, samples * classes, 2)
    y = zeros(Int, samples * classes)
    for class_number in 1:classes
        ix = (samples * (class_number - 1) + 1):(samples * class_number)
        r = LinRange(0.0f0, 1.0f0, samples)
        t = LinRange((class_number - 1) * 4, class_number * 4, samples) .+ randn(Float32, samples) .* 0.2f0
        X[ix, 1] = r .* sin.(t .* 2.5f0)
        X[ix, 2] = r .* cos.(t .* 2.5f0)
        y[ix] .= class_number
    end
    return X, y
end

# --- Dense Layer ---
mutable struct Layer_Dense
    weights::Matrix{Float32}
    biases::Matrix{Float32}
    inputs::Matrix{Float32}
    output::Matrix{Float32}
    dweights::Matrix{Float32}
    dbiases::Matrix{Float32}
    dinputs::Matrix{Float32}

    function Layer_Dense(n_inputs::Int, n_neurons::Int)
        weights = 0.01f0 .* randn(Float32, n_inputs, n_neurons)
        biases = zeros(Float32, 1, n_neurons)
        new(weights, biases, Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0))
    end
end

function forward(layer::Layer_Dense, inputs::Matrix{Float32})
    layer.inputs = inputs
    layer.output = inputs * layer.weights .+ layer.biases
end

function backward(layer::Layer_Dense, dvalues::Matrix{Float32})
    layer.dweights = layer.inputs' * dvalues
    layer.dbiases = sum(dvalues, dims=1)
    layer.dinputs = dvalues * layer.weights'
end

# --- ReLU Activation ---
mutable struct Activation_ReLU
    inputs::Matrix{Float32}
    output::Matrix{Float32}
    dinputs::Matrix{Float32}

    Activation_ReLU() = new(Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0))
end

function forward(activation::Activation_ReLU, inputs::Matrix{Float32})
    activation.inputs = inputs
    activation.output = max.(0.0f0, inputs)
end

function backward(activation::Activation_ReLU, dvalues::Matrix{Float32})
    activation.dinputs = copy(dvalues)
    activation.dinputs[activation.inputs .<= 0.0f0] .= 0.0f0
end

# --- Combined Softmax and Loss ---
mutable struct Activation_Softmax_Loss_CategoricalCrossentropy
    output::Matrix{Float32}
    dinputs::Matrix{Float32}

    Activation_Softmax_Loss_CategoricalCrossentropy() = new(Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0))
end

function forward(combo::Activation_Softmax_Loss_CategoricalCrossentropy, inputs::Matrix{Float32}, y_true::Vector{Int})
    # Softmax activation
    exp_values = exp.(inputs .- maximum(inputs, dims=2))
    combo.output = exp_values ./ sum(exp_values, dims=2)

    # Categorical Cross-Entropy Loss
    n_samples = size(combo.output, 1)
    probs_clipped = clamp.(combo.output, 1e-7, 1 - 1e-7)
    correct_confidences = [probs_clipped[i, y_true[i]] for i in 1:n_samples]
    
    return mean(-log.(correct_confidences))
end

function backward(combo::Activation_Softmax_Loss_CategoricalCrossentropy, y_true::Vector{Int})
    n_samples = size(combo.output, 1)
    combo.dinputs = copy(combo.output)

    for (i, true_class_idx) in enumerate(y_true)
        combo.dinputs[i, true_class_idx] -= 1
    end
    combo.dinputs = combo.dinputs ./ n_samples
end

# --- SGD Optimizer with Learning Rate Decay ---
mutable struct Optimizer_SGD
    learning_rate::Float64
    current_learning_rate::Float64
    decay::Float64
    iterations::Int

    function Optimizer_SGD(learning_rate::Float64=1.0, decay::Float64=0.0)
        new(learning_rate, learning_rate, decay, 0)
    end
end

function pre_update_params(optimizer::Optimizer_SGD)
    if optimizer.decay > 0.0
        optimizer.current_learning_rate = optimizer.learning_rate * (1.0 / (1.0 + optimizer.decay * optimizer.iterations))
    end
end

function update_params(optimizer::Optimizer_SGD, layer::Layer_Dense)
    layer.weights .-= optimizer.current_learning_rate .* layer.dweights
    layer.biases .-= optimizer.current_learning_rate .* layer.dbiases
end

function post_update_params(optimizer::Optimizer_SGD)
    optimizer.iterations += 1
end

# --- Main Training Script ---
function main()
    println("--- Initializing Pipeline ---")
    
    # 1. Generate Data and Create Model Instances
    X, y = spiral_data(100, 3)
    dense1 = Layer_Dense(2, 64)
    activation1 = Activation_ReLU()
    dense2 = Layer_Dense(64, 3)
    loss_activation = Activation_Softmax_Loss_CategoricalCrossentropy()
    
    # 2. Instantiate the optimizer with a decay rate using POSITIONAL arguments
    optimizer = Optimizer_SGD(0.7, 1e-4)
    
    println("\n--- Starting Training Loop ---")
    @printf "%-10s %-15s %-15s %-20s\n" "Epoch" "Accuracy" "Loss" "Learning Rate"
    println(repeat("-", 60))

    # 3. Training Loop
    for epoch in 0:10000
        # Forward Pass
        forward(dense1, X)
        forward(activation1, dense1.output)
        forward(dense2, activation1.output)
        loss = forward(loss_activation, dense2.output, y)
        
        # Calculate Accuracy
        predictions = [argmax(row) for row in eachrow(loss_activation.output)]
        accuracy = mean(predictions .== y)
        
        # Print progress every 1000 epochs
        if epoch % 100 == 0
            @printf "%-10d %-15.3f %-15.3f %-20.7f\n" epoch accuracy loss optimizer.current_learning_rate
        end
        
        # Backward Pass
        backward(loss_activation, y)
        backward(dense2, loss_activation.dinputs)
        backward(activation1, dense2.dinputs)
        backward(dense1, activation1.dinputs)
        
        # Optimization
        pre_update_params(optimizer)
        update_params(optimizer, dense1)
        update_params(optimizer, dense2)
        post_update_params(optimizer)
    end
    
    println(repeat("-", 60))
    println("Training complete.")
end

# Run the main function
main()


--- Initializing Pipeline ---

--- Starting Training Loop ---
Epoch      Accuracy        Loss            Learning Rate       
------------------------------------------------------------
0          0.250           1.099           0.7000000           
100        0.443           1.072           0.6931379           
200        0.450           1.048           0.6863418           
300        0.447           1.045           0.6796776           
400        0.440           1.044           0.6731416           
500        0.440           1.044           0.6667302           
600        0.433           1.043           0.6604397           
700        0.440           1.043           0.6542668           
800        0.440           1.042           0.6482082           
900        0.467           1.041           0.6422608           
1000       0.467           1.038           0.6364215           
1100       0.470           1.035           0.6306874           
1200       0.463           1.031           0.